# ETL del archivo en crudo `cast.parquet`

## Librerías

In [23]:
import os
import ast
import gc

import pandas as pd

## Extracción

In [24]:
url = "https://github.com/FranciscoHugoLezik/Movies_data/blob/main/credits/cast.parquet?raw=true"

cast = pd.read_parquet(
    url, 
    engine='fastparquet'
    )

In [25]:
cast.head()

,cast,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...",11862


Valor en la columna 'cast' de la primera fila. Es una cadena con la forma de una lista de diccionarios.

In [26]:
cast['cast'].iloc[0]

"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}, {'cast_id': 17, 'character': 'Slinky Dog (voice)', 'credit_id': '52fe4284c3a36847f8024fa1', 'gender': 2, 'id': 12899, 'name': 'Jim Varney', 'order': 3, 'profile_path': '/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg'}, {'cast_id': 18, 'character': 'Rex (voice)', 'credit_id': '52fe4284c3a36847f8024fa5', 'gender': 2, 'id': 12900, 'name': 'Wallace Shawn', 'order': 4, 'profile_path': '/oGE6JqPP2xH4t

Las columnas estan completas.

In [27]:
cast.isnull().sum()

cast    0
id      0
dtype: int64

## Transformación

### Borrar duplicados en la columna 'id'

Hay valores duplicados en la columna 'id'.

In [28]:
cast['id'].duplicated(
    keep='first').sum()

44

Se eliminan los duplicados.

In [29]:
cast.drop_duplicates(
    subset='id', 
    inplace=True
    )

Hay valores unicos.

In [30]:
cast['id'].duplicated(
    keep='first').sum()

0

### Renombrar la columna 'id'

Se hace esto porque dentro de los valores anidados hay una clave llamada 'id' y para, mas adelante, poder hacer un join con el dataset 'movies.parquet'.

In [31]:
cast.rename(
    columns={'id': 'movie_id'}, 
    inplace=True
    )

Se cambio el nombre.

In [32]:
cast.columns

Index(['cast', 'movie_id'], dtype='object')

### Eliminar listas vacias en la columna 'cast'

Se van a eliminar listas vacías de la columna 'cast' para achicar el tamaño del dataset.

Hay listas vacías.

In [33]:
cast[cast['cast'] == "[]"]

,cast,movie_id
137,[],124639
240,[],43475
393,[],42981
438,[],24257
595,[],124472
...,...,...
45447,[],455661
45452,[],44330
45458,[],122036
45462,[],276895


In [34]:
len(cast[cast['cast'] == "[]"])

2414

Se eliminan las listas vacías.

In [35]:
cast = cast.query('cast != "[]"')

Las listas vacías estan eliminadas.

In [36]:
len(cast[cast['cast'] == "[]"])

0

### Desanidar la columna 'cast'

Se convierte a las cadenas en listas de diccionarios.

In [37]:
cast['cast'] = cast['cast'].apply(
    ast.literal_eval)

Se separan los elementos de las listas en filas.

In [38]:
cast_en_filas = cast.explode('cast').reset_index(drop=True)

In [39]:
cast_en_filas

,cast,movie_id
0,"{'cast_id': 14, 'character': 'Woody (voice)', ...",862
1,"{'cast_id': 15, 'character': 'Buzz Lightyear (...",862
2,"{'cast_id': 16, 'character': 'Mr. Potato Head ...",862
3,"{'cast_id': 17, 'character': 'Slinky Dog (voic...",862
4,"{'cast_id': 18, 'character': 'Rex (voice)', 'c...",862
...,...,...
562039,"{'cast_id': 2, 'character': '', 'credit_id': '...",227506
562040,"{'cast_id': 3, 'character': '', 'credit_id': '...",227506
562041,"{'cast_id': 4, 'character': '', 'credit_id': '...",227506
562042,"{'cast_id': 5, 'character': '', 'credit_id': '...",227506


Se convierten las llaves en columnas.

In [40]:
cast_en_columnas = pd.json_normalize(
    cast_en_filas['cast'])

In [41]:
cast_en_columnas

,cast_id,character,credit_id,gender,id,name,order,profile_path
0,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
1,15,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,2,12898,Tim Allen,1,/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg
2,16,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2,7167,Don Rickles,2,/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg
3,17,Slinky Dog (voice),52fe4284c3a36847f8024fa1,2,12899,Jim Varney,3,/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg
4,18,Rex (voice),52fe4284c3a36847f8024fa5,2,12900,Wallace Shawn,4,/oGE6JqPP2xH4tNORKNqxbNPYi7u.jpg
...,...,...,...,...,...,...,...,...
562039,2,,52fe4ea59251416c7515d7d5,2,544742,Iwan Mosschuchin,0,None
562040,3,,52fe4ea59251416c7515d7d9,1,1090923,Nathalie Lissenko,1,None
562041,4,,52fe4ea59251416c7515d7dd,2,1136422,Pavel Pavlov,2,None
562042,5,,52fe4ea59251416c7515d7e1,0,1261758,Aleksandr Chabrov,3,None


### Crear un nuevo dataframe 'cast'

Se crea un dataframe con la columna 'movie_id' junto con las nuevas columnas.

In [42]:
cast = cast_en_filas.drop(
    columns='cast').join(
        cast_en_columnas)

Se eliminan los siguientes objetos para liberar memoria.

In [43]:
del cast_en_filas
del cast_en_columnas
gc.collect()

363

## Exploración

Se explora el dataframe.

In [44]:
cast

,movie_id,cast_id,character,credit_id,gender,id,name,order,profile_path
0,862,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
1,862,15,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,2,12898,Tim Allen,1,/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg
2,862,16,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2,7167,Don Rickles,2,/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg
3,862,17,Slinky Dog (voice),52fe4284c3a36847f8024fa1,2,12899,Jim Varney,3,/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg
4,862,18,Rex (voice),52fe4284c3a36847f8024fa5,2,12900,Wallace Shawn,4,/oGE6JqPP2xH4tNORKNqxbNPYi7u.jpg
...,...,...,...,...,...,...,...,...,...
562039,227506,2,,52fe4ea59251416c7515d7d5,2,544742,Iwan Mosschuchin,0,None
562040,227506,3,,52fe4ea59251416c7515d7d9,1,1090923,Nathalie Lissenko,1,None
562041,227506,4,,52fe4ea59251416c7515d7dd,2,1136422,Pavel Pavlov,2,None
562042,227506,5,,52fe4ea59251416c7515d7e1,0,1261758,Aleksandr Chabrov,3,None


Primera fila.

In [45]:
cast.iloc[0]

movie_id                                     862
cast_id                                       14
character                          Woody (voice)
credit_id               52fe4284c3a36847f8024f95
gender                                         2
id                                            31
name                                   Tom Hanks
order                                          0
profile_path    /pQFoyx7rp09CJTAb932F2g8Nlho.jpg
Name: 0, dtype: object

La columna 'name' esta completa.

In [46]:
cast['name'].isnull().sum()

0

## Transformación de los datos desanidados

### Eliminar las columnas innecesarias

Se las elimina porque son inutiles para la funcion get_actor.

Columnas innecesarias.

In [47]:
innecesarias = [
    'cast_id', 
    'credit_id', 
    'gender', 
    'id', 
    'order', 
    'profile_path'
]

Las columnas innecesarias son eliminadas.

In [48]:
cast.drop(
    columns=innecesarias, 
    inplace=True
    )

Las columnas innecesarias estan eliminadas.

In [49]:
set(cast.columns).isdisjoint(set(innecesarias))

True

Se ven las columnas que quedan.

In [50]:
for columna in cast.columns:
    print(columna)

movie_id
character
name


### Cambiar el tipo de la columna 'movie_id'

La columna 'movie_id' tiene etiquetas. Entonces se lo cambia al tipo object. El resto de los datos tienen el tipo correcto que es object.

Tipos de las columnas:

In [51]:
cast.dtypes

movie_id      int64
character    object
name         object
dtype: object

* Columna 'movie_id':

In [52]:
cast['movie_id'].dtype

dtype('int64')

In [53]:
cast['movie_id'] = cast['movie_id'].astype(str)

In [54]:
cast['movie_id'].dtype

dtype('O')

Tipos de las columnas con la modificación:

In [55]:
cast.dtypes

movie_id     object
character    object
name         object
dtype: object

### Resetear el índice

El índice actual:

In [56]:
cast.index

RangeIndex(start=0, stop=562044, step=1)

Se resetea el índice:

In [57]:
cast.reset_index(
    drop=True, 
    inplace=True
    )

El índice actualizado:

In [58]:
cast.index

RangeIndex(start=0, stop=562044, step=1)

### Última revisión

In [59]:
cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 562044 entries, 0 to 562043
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   movie_id   562044 non-null  object
 1   character  562044 non-null  object
 2   name       562044 non-null  object
dtypes: object(3)
memory usage: 12.9+ MB


## Carga

In [60]:
ruta_actual = os.getcwd()

ruta_actual

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\notebooks\\ETL'

In [61]:
ruta_del_proyecto = os.path.dirname(
    os.path.dirname(
        ruta_actual))

ruta_del_proyecto

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas'

In [62]:
ruta_a_exportar = os.path.join(
    ruta_del_proyecto, 
    'data', 
    'ETL', 
    'cast.parquet')

ruta_a_exportar

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\data\\ETL\\cast.parquet'

In [63]:
cast.to_parquet(ruta_a_exportar)

Se elimina el dataframe para liberar memoria.

In [64]:
del cast
gc.collect()

45